# Phase B v2 — Dataset (7-class, 9-band, enhanced-MK targets)
Generates the v2 training set once the enhanced-MK physics (Phase A v2) is
implemented in `SIM/phase_a.py` behind `PHYSICS_VARIANT='v2.0'`.

Changes from v1: **7 classes** (glass un-fold), **9-band frequency union**
(5 scanner + 4 indoor-Tx), targets from the enhanced-MK loss model. Storage,
splits (by position), and audit are unchanged.

In [ ]:
import os
if not os.path.exists('indoor-walk-test'):
    os.system('git clone --depth 1 https://github.com/cgm2179/indoor-walk-test.git')
%cd indoor-walk-test
import importlib.util, json, numpy as np
spec = importlib.util.spec_from_file_location('pa', 'SIM/phase_a.py')
pa = importlib.util.module_from_spec(spec); spec.loader.exec_module(pa)
print('PHYSICS_VARIANT =', pa.PHYSICS_VARIANT, '(must be "v2.0" for a real v2 dataset)')
print('v2 freq union:', pa.V2['freqs_mhz'])

## Prerequisite
This notebook is a scaffold: it will produce a correct v2 dataset **only after**
`SIM/phase_a.py` implements `PHYSICS_VARIANT='v2.0'` (7-class grid + γ loss +
enhancements) and `SIM/phase_b_dataset.py` reads the 9-band `V2['freqs_mhz']`.
Until then it runs against v1 physics and is for pipeline testing only.

In [ ]:
# v2 dataset generation reuses phase_b with the v2 frequency list. When
# phase_a v2 is ready, set PHYSICS_VARIANT and point FREQS_MHZ at V2.
# For now, demonstrate the 9-band anchor set and the shard plan.
freqs = pa.V2['freqs_mhz']
N_POS = 2500
print(f'{N_POS} positions x {len(freqs)} freqs = {N_POS*len(freqs)} samples '
      f'({(N_POS*len(freqs)*256*448*2)/1e9:.1f} GB float16 targets)')
print('run (once v2 physics is in): '
      '!PHYSICS_VARIANT=v2.0 python SIM/phase_b_dataset.py')
# TODO: parameterize phase_b_dataset.py to read pa.V2["freqs_mhz"] and the
# 7-class grid when PHYSICS_VARIANT=="v2.0".

In [ ]:
# Audit is identical to v1: split disjointness, all-freqs-together, histogram,
# clip fractions (expect the high-clip % to RISE with the 20 dB low-E facade).
print('after generation: !python SIM/phase_b_dataset.py --audit')